# Curs 1 — Primul apel către un LLM
## LLM-uri, API-uri și parametri de generare
În acest notebook facem primul apel către un model Gemini folosind Python.
Scopul nu este să construim încă un agent, ci să înțelegem structura minimă a unui apel către un LLM:
- modelul folosit
- mesajele trimise către model
- parametrii de generare
- outputul primit
La finalul notebook-ului trebuie să avem:
1. API key configurată corect în `.env`
2. un prim apel funcțional către Gemini
3. un rezumat neutru generat de model
4. un mic experiment cu `temperature`
5. un output salvat împreună cu parametrii folosiți
---


## 0. Instalare pachete

In [1]:
# Rulează o singură dată. Dacă ești în Google Colab, sari peste acest pas.
%pip install -q openai python-dotenv requests


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Instalare biblioteci
Folosim biblioteca `openai`, dar nu folosim un model OpenAI.
Folosim Gemini prin endpoint-ul compatibil cu formatul OpenAI.
-  sintaxa `messages = [{"role": "...", "content": "..."}]` este foarte răspândită și ne va ajuta mai târziu să lucrăm și cu OpenRouter, DeepSeek sau alte modele compatibile.

## 2. Configurarea cheii API

https://aistudio.google.com/api-keys

Cheia API nu se scrie direct în notebook.
O salvăm într-un fișier local numit `.env`.
Fișierul `.env` trebuie să conțină:
```text
GEMINI_API_KEY=cheia_ta_aici

!Important:

nu urca niciodată .env pe GitHub
în repository trebuie să existe doar .env.example
dacă ai publicat cheia accidental, șterge cheia din Google AI Studio și generează alta

In [3]:
!pip install dotenv

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)

   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 2/2 [dotenv]




[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import sys
!{sys.executable} -m pip install google-genai

  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.13.3-py3-none-any.whl.metadata (108 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.3-cp314-cp314-win_amd64.whl.metadata (6.7 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/790.4 kB ? eta -:--:--
   ---------------------------------------- 790.4/790.4 kB 6.6 MB/s  0:00:00
Using cached anyio-4.13.0-py3-none-any.whl 


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import sys
!{sys.executable} -m pip install openai

  Using cached openai-2.33.0-py3-none-any.whl.metadata (31 kB)
  Using cached jiter-0.14.0-cp314-cp314-win_amd64.whl.metadata (5.3 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached openai-2.33.0-py3-none-any.whl (1.2 MB)
Using cached jiter-0.14.0-cp314-cp314-win_amd64.whl (202 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)

   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   -------------------------- ------------- 2/3 [openai]
   -------------------------- ------------- 2/3 [openai]
   -------------------------- ------------- 2/3 [openai]
   -------------------------- ------------- 2/3 [openai]
   -----------------------


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from google import genai


## 3. Creăm clientul API
Clientul este obiectul prin care trimitem cereri către model.
Aici avem trei lucruri importante:
- `api_key`: cheia personală
- `base_url`: adresa endpoint-ului Gemini compatibil cu formatul OpenAI
- `model`: modelul pe care îl vom folosi în apel

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Missing GEMINI_API_KEY. Add it to your .env file.")

In [3]:
from openai import OpenAI

client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

MODEL = "gemini-2.5-flash"

print("Client configurat.")
print("Model selectat:", MODEL)

Client configurat.
Model selectat: gemini-2.5-flash


## 4. Primul apel minimal
Începem cu cel mai simplu caz: trimitem o singură întrebare către model și primim un răspuns.
Aici nu folosim încă `system message`, nu definim un rol special și nu construim o conversație.
Vrem doar să verificăm că:
- cheia API funcționează
- modelul răspunde
- înțelegem forma minimă a unui apel




In [8]:
echo %OPENAI_API_KEY%   

your-key-here   


In [10]:
client = OpenAI()

In [18]:
print(GEMINI_API_KEY)


your-key-here


In [4]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Say hello."}
    ]
)
print(response.choices[0].message.content)

Hello!


In [5]:
# raspunsul modelului 
response.choices[0].message.content

'Hello!'

In [6]:
response

ChatCompletion(id='x0X2aeDhH9q0xN8PoMuZ0AI', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello!', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1777747400, model='gemini-2.5-flash', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=2, prompt_tokens=4, total_tokens=29, completion_tokens_details=None, prompt_tokens_details=None))

- ChatCompletion(...) = obiectul complet returnat de API. Nu este doar textul, ci răspunsul + metadata.
- id='JRvvaaXoMueynsEPusrdsQM' = identificator unic al apelului. Util pentru debugging/loguri.
- object='chat.completion' = tipul obiectului returnat. Confirmă că este un răspuns de tip chat completion.
- created=1777277734 = timestamp intern al momentului când a fost creat răspunsul.
- model='gemini-2.5-flash' = modelul care a generat răspunsul.
- choices=[...] = lista de răspunsuri generate. De obicei folosim primul răspuns: choices[0].
- index=0 = poziția răspunsului în lista choices. Aici este primul și singurul răspuns.
- message=ChatCompletionMessage(...) = mesajul generat de model.
- message.content='Hello!' = textul efectiv al răspunsului. Îl accesezi cu:
- response.choices[0].message.content
- message.role='assistant' = rolul mesajului generat. Modelul răspunde ca assistant.
- finish_reason='stop' = modelul s-a oprit normal. Nu a fost tăiat de limită.
- logprobs=None = nu ai cerut probabilități pentru tokeni.
- refusal=None = modelul nu a refuzat cererea.
- annotations=None = nu există adnotări suplimentare returnate.
- audio=None = nu există output audio.
- function_call=None = modelul nu a cerut apelarea unei funcții vechi-style.
- tool_calls=None = modelul nu a cerut folosirea unui tool.
- service_tier=None = nu apare un nivel special de serviciu raportat.
- system_fingerprint=None = providerul nu a returnat o amprentă internă a sistemului.
- usage=CompletionUsage(...) = informații despre tokeni consumați.
- prompt_tokens=4 = tokenii trimiși către model în cerere.
- completion_tokens=2 = tokenii generați de model în răspuns.
- total_tokens=31 = totalul raportat de provider. La Gemini prin compatibilitate OpenAI poate include overhead intern, deci nu trebuie tratat mereu ca simpla sumă prompt + completion.
- completion_tokens_details=None și prompt_tokens_details=None = nu există detalii suplimentare despre tokeni.

In [7]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": ""
            "You are a clear teacher explaining AI to sociology students."
        },
        {
            "role": "user",
            "content": "Explain what a large language model is in 3 simple sentences."
        }
    ],
    temperature=0.3
)

print(response.choices[0].message.content)

A Large Language Model (LLM) is a type of artificial intelligence program trained on an enormous amount of text data from the internet and books. It learns to understand, generate, and predict human language by recognizing complex patterns, grammar, and context within that data. This allows it to perform tasks like answering questions, writing essays, summarizing information, and even translating languages in a remarkably human-like way.


## 5. Mai multe modele

In [8]:
models_to_test = [
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
]

for model_name in models_to_test:
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "user", "content": "Explain what an LLM is in one simple sentence."}
        ]
    )
    print("=" * 60)
    print("MODEL:", model_name)
    print(response.choices[0].message.content)

MODEL: gemini-2.5-flash
An LLM is a large artificial intelligence model trained on massive amounts of text to understand and generate human-like language.
MODEL: gemini-2.5-flash-lite
An LLM is a type of AI that learns patterns from vast amounts of text data to understand and generate human-like language.


In [9]:
response

ChatCompletion(id='40X2af_lL7eBkdUPzbHuoQE', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='An LLM is a type of AI that learns patterns from vast amounts of text data to understand and generate human-like language.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1777747427, model='gemini-2.5-flash-lite', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=26, prompt_tokens=12, total_tokens=38, completion_tokens_details=None, prompt_tokens_details=None))

## 6. Exercițiu: rezumat neutru al unei știri
Vom da modelului un text scurt și îi cerem să producă un rezumat neutru.
Regulă:
- maximum 60 de cuvinte
- fără opinie personală
- fără exagerări
- fără informații care nu apar în text

In [10]:
news_text = """
Guvernul a anunțat un nou program de finanțare pentru modernizarea transportului public în orașele mari.
Programul include achiziția de autobuze electrice, modernizarea stațiilor și extinderea sistemelor de ticketing digital.
Reprezentanții ministerului spun că scopul este reducerea poluării și creșterea accesului la transport public.
Unele organizații civice au cerut criterii clare de alocare a fondurilor și transparență în implementare.
"""

prompt = f"""
Rezuma textul de mai jos în maximum 60 de cuvinte.
Rezumatul trebuie să fie neutru, clar și factual.
Nu adăuga informații care nu apar în text.

TEXT:
{news_text}
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You summarize public-interest texts in a neutral and factual way."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2,
    
)

summary = response.choices[0].message.content

print(summary)

Guvernul a anunțat un program de finanțare pentru modernizarea transportului public în orașele mari. Acesta include achiziția de autobuze electrice, modernizarea stațiilor și extinderea sistemelor de ticketing digital. Scopul declarat este reducerea poluării și creșterea accesului. Organizațiile civice au cerut criterii clare de alocare a fondurilor și transparență în implementare.


## 7. Ce face `temperature`?
`temperature` controlează cât de variat este răspunsul.
- valori mici: răspunsuri mai stabile, mai previzibile
- valori mari: răspunsuri mai creative, dar uneori mai puțin precise
Pentru analiză socială, clasificare și rezumate factuale, începem de obicei cu valori mici: `0`, `0.2`, `0.3`.
Pentru generare creativă putem folosi valori mai mari.

In [11]:
test_prompt = """
Descrie în două propoziții ce este un model lingvistic mare.
Explică pentru studenți de sociologie, fără jargon tehnic.
"""

temperatures = [0.0, 0.7, 1.5]

for temp in temperatures:
    response = client.chat.completions.create(
        model="gemini-2.5-flash-lite",
        messages=[
            {
                "role": "system",
                "content": "You explain AI concepts clearly and briefly."
            },
            {
                "role": "user",
                "content": test_prompt
            }
        ],
        temperature=temp,
        max_tokens=120
    )

    print("=" * 80)
    print("TEMPERATURE:", temp)
    print(response.choices[0].message.content)

TEMPERATURE: 0.0
Un model lingvistic mare este ca un asistent super inteligent care a citit o cantitate uriașă de texte, de la cărți la articole de pe internet. Datorită acestei "lecturi" extinse, el poate înțelege și genera limbaj uman, ajutându-ne să scriem, să rezumăm informații sau chiar să purtăm conversații.
TEMPERATURE: 0.7
Un model lingvistic mare este ca un asistent digital care a "citit" o cantitate uriașă de text, de la cărți la articole de pe internet, și a învățat cum funcționează limbajul uman. Datorită acestei "lecturi", poate înțelege și genera text, răspunzând la întrebări sau scriind povestiri, la fel cum un sociolog ar analiza și interpreta informații despre societate.
TEMPERATURE: 1.5
Un model lingvistic mare este ca un scrib virtual foarte bine pregătit, care a citit o cantitate imensă de texte, precum cărți, articole și conversații online, pentru a învăța cum să folosească limbajul. Datorită acestei "lecturi" extinse, poate să înțeleagă și să genereze text asemănă

## 8. Mini-interpretare
Completați după rulare:
1. Care răspuns a fost cel mai stabil?
2. Care răspuns a fost cel mai creativ?
3. Care răspuns este mai potrivit pentru cercetare academică?
4. Ce valoare de `temperature` ați folosi pentru rezumate neutre?
TODO:
Scrieți aici observațiile voastre.

#1. Cel mai stabil răspuns TEMPERATURE 0.0: O temperatură de $0$ face modelul "determinist". Asta înseamnă că modelul va alege întotdeauna cuvintele cu cea mai mare probabilitate statistică. Dacă rulezi același prompt de mai multe ori cu temperatura 0, vei obține (aproape) de fiecare dată același text, fără variații.

#2. TEMPERATURE 1.5 este cel mai creativ: Pe măsură ce valoarea temperaturii crește, modelul începe să "riște" mai mult, alegând cuvinte care nu sunt neapărat primele în topul probabilităților. La 1.5, limbajul devine mai variat și folosește analogii mai puțin comune (precum "student extrem de studios"), dar crește și riscul de a genera text incoerent sau ilogic.

#3. TEMPERATURE 0.7: În cercetarea academică, ai nevoie de un echilibru între precizie și cursivitate. Răspunsul de la 0.7 este definit ca fiind un "program de calculator", ceea ce este o descriere tehnică corectă și obiectivă. Cel de la 0.0 este de asemenea foarte bun pentru că oferă o definiție directă, fără înflorituri literare.

#4. Pentru rezumate neutre, aș folosi o valoare între 0.1 și 0.3:Un rezumat trebuie să fie fidel textului sursă și să nu adauge informații de la "sine" (halucinații).O temperatură scăzută asigură faptul că modelul rămâne concentrat pe fapte și folosește un limbaj standard, lipsit de subiectivitate sau figuri de stil inutile.Valoarea 0.0 poate fi uneori prea repetitivă, așa că un 0.2 oferă acea mică variație necesară pentru ca textul să fie lizibil, păstrându-și în același timp neutralitatea strictă.


## 9. Funcție simplă pentru apeluri repetate


In [12]:
MODEL = "gemini-2.5-flash-lite"
def ask_model(
    user_message,
    system_message="You are a helpful assistant.",
    model=MODEL,
    temperature=0.3,
    max_tokens=300
):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": system_message
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )

    return response.choices[0].message.content

In [13]:
answer = ask_model(
    user_message="Explain the difference between an LLM and an AI agent in 4 bullet points.",
    system_message="You explain AI concepts to sociology students using simple language.",
    temperature=0.3,
    max_tokens=250
)

print(answer)

Here's the difference between an LLM and an AI agent, explained simply for sociology students:

*   **LLM (Large Language Model): The Smart Talker.** Think of an LLM like a super-powered chatbot or a really, really good writer. Its main job is to understand and generate human-like text. It's great at answering questions, summarizing things, writing stories, or even translating languages, but it doesn't *do* things in the real world on its own.

*   **AI Agent: The Doer.** An AI agent is like a digital assistant that can not only understand and talk (often using an LLM for that part!) but also *act* on your behalf. It has goals and can take steps to achieve them, like booking an appointment, searching the internet for specific information and then using it, or even controlling other software.

*   **LLM is a Tool, Agent is a System.** An LLM is a core component, like the engine in a car. An AI agent is the whole car – it uses the engine (LLM) but also has wheels, a steering wheel, and a

## 10. Salvăm outputul și parametrii


Trebuie să știm:
- ce model am folosit
- ce prompt am trimis
- ce parametri am setat
- ce răspuns am primit


In [14]:
import json
from datetime import datetime
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

experiment = {
    "timestamp": datetime.now().isoformat(),
    "model": MODEL,
    "temperature": 0.2,
    "max_tokens": 120,
    "task": "neutral_news_summary",
    "input_text": news_text,
    "output_text": summary
}

output_path = output_dir / "c1_first_summary.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(experiment, f, ensure_ascii=False, indent=2)

print("Experiment salvat în:", output_path)

Experiment salvat în: outputs\c1_first_summary.json


## 11. Verificăm fișierul salvat
Acum citim fișierul înapoi, ca să vedem că rezultatul a fost salvat corect.

In [15]:
with open(output_path, "r", encoding="utf-8") as f:
    saved_experiment = json.load(f)

saved_experiment

{'timestamp': '2026-05-02T21:44:26.612366',
 'model': 'gemini-2.5-flash-lite',
 'temperature': 0.2,
 'max_tokens': 120,
 'task': 'neutral_news_summary',
 'input_text': '\nGuvernul a anunțat un nou program de finanțare pentru modernizarea transportului public în orașele mari.\nProgramul include achiziția de autobuze electrice, modernizarea stațiilor și extinderea sistemelor de ticketing digital.\nReprezentanții ministerului spun că scopul este reducerea poluării și creșterea accesului la transport public.\nUnele organizații civice au cerut criterii clare de alocare a fondurilor și transparență în implementare.\n',
 'output_text': 'Guvernul a anunțat un program de finanțare pentru modernizarea transportului public în orașele mari. Acesta include achiziția de autobuze electrice, modernizarea stațiilor și extinderea sistemelor de ticketing digital. Scopul declarat este reducerea poluării și creșterea accesului. Organizațiile civice au cerut criterii clare de alocare a fondurilor și transpa

## 12. Exercițiu scurt pentru studenți
Alegeți un text scurt, de 5–8 rânduri, dintr-o sursă publică.
Poate fi despre transport, sănătate, educație, mediu sau costul vieții.
Completați variabila de mai jos și generați un rezumat neutru.
Apoi salvați rezultatul.

In [16]:
student_text = """
Datele FePAL (Federația Părinților și Aparținătorilor Legali) arată rezultate îngrijorătoare, în contextul în care în România nu există încă un cadru sau o direcție clară privind utilizarea inteligenței artificiale.

98% dintre elevii de peste 16 ani folosesc AI, iar 1 din 3 utilizează această tehnologie fără să o înțeleagă pe deplin. 
Deși tinerii sunt deschiși la integrarea inteligenței artificiale în procesul de învățare, ei recunosc și riscurile asociate unei utilizări nestructurate.
Astfel, 81% dintre ei consideră că AI-ul poate duce la copiere sau dependență.

Aceste tendințe se vor reflecta și pe piața muncii. 
Utilizarea inteligenței artificiale va deveni unul dintre criteriile de selecție la locul de muncă al viitorului, iar școala românească trebuie să pregătească tinerii pentru era tehnologiei.
"""

student_prompt = f"""
Rezuma textul de mai jos în maximum 60 de cuvinte.
Rezumatul trebuie să fie neutru, clar și factual.
Nu adăuga informații care nu apar în text.

TEXT:
{student_text}
"""

student_summary = ask_model(
    user_message=student_prompt,
    system_message="You summarize public-interest texts in a neutral and factual way.",
    temperature=0.2,
    max_tokens=120
)

print(student_summary)

98% dintre elevii români peste 16 ani folosesc inteligența artificială, 33% fără a o înțelege complet. Elevii recunosc riscurile de copiere și dependență, dar sunt deschiși la integrarea AI în învățare. FePAL subliniază lipsa unui cadru clar pentru utilizarea AI în România și necesitatea pregătirii tinerilor pentru piața muncii viitoare, unde AI va fi un criteriu de selecție.


In [17]:
student_experiment = {
    "timestamp": datetime.now().isoformat(),
    "model": MODEL,
    "temperature": 0.2,
    "max_tokens": 120,
    "task": "student_neutral_summary",
    "input_text": student_text,
    "output_text": student_summary
}

student_output_path = output_dir / "c1_student_summary.json"

with open(student_output_path, "w", encoding="utf-8") as f:
    json.dump(student_experiment, f, ensure_ascii=False, indent=2)

print("Experiment salvat în:", student_output_path)

Experiment salvat în: outputs\c1_student_summary.json


## 13. Checklist pentru finalul Cursului 1
La finalul acestui notebook trebuie să aveți:
- API key funcțională
- `.env` local configurat
- primul apel către Gemini rulat cu succes
- un rezumat neutru generat
- experimentul cu `temperature` rulat
- output salvat în `outputs/`
- `.env` adăugat în `.gitignore`
Pentru repository:
- `README.md`
- `.env.example`
- `.gitignore`
- notebook-ul de Curs 1
- fără cheia API în GitHub

## 14. Ce luăm mai departe în Cursul 2
Acum știm să apelăm un model și să controlăm minim răspunsul.
În Cursul 2 vom compara modele:
- același prompt
- modele diferite
- criterii simple de evaluare: claritate, factualitate, neutralitate, viteză
Întrebarea următoare nu mai este „cum chemăm modelul?”, ci „ce model alegem pentru proiect?”.